# ISMB 2026 Tutorial — Part 3
# Introduction to Visual-Omics Foundation Models: OmiCLIP

**Instructor:** Guangyu Wang 

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain the motivation for a contrastive vision–language model built on paired H&E + transcriptomics data.
2. Describe the OmiCLIP training objective and the structure of the image / transcriptomics encoders.
3. Load a pretrained OmiCLIP checkpoint and compute embeddings for tissue patches and gene-expression spots.
4. Perform cross-modal retrieval (image → expression, expression → image) on a Visium slide.
5. Read a UMAP of the joint embedding space and interpret what "alignment" means here.

---

## 0. Brief recap of the slides

OmiCLIP is a CLIP-style contrastive foundation model that aligns two modalities collected on the **same** 10x Visium spots:

* a **tissue patch** centered on a Visium spot (an H&E image crop ~224×224 px), encoded by a vision transformer;
* the **transcriptomic profile** of that spot, encoded by a transformer over gene-expression tokens (top-k highly variable genes per spot, ranked-token representation).

The two encoders are trained with an **InfoNCE / symmetric cross-entropy loss** that pulls together embeddings from the same spot and pushes apart embeddings from different spots in a large mini-batch. After pretraining on >2M Visium spot pairs across >100 tissues, the encoders produce a shared latent space that is reused, frozen, in every downstream task we will see in Part 4.


> 🔁 **Why contrastive?** It only needs *paired* data, not labels. Every Visium slide gives us tens of thousands of free positive pairs (the H&E patch and the expression vector at the same spot) — perfect for foundation-model-scale pretraining.


## 1. Environment setup

The notebook is designed to run on a single GPU (≥16 GB VRAM recommended) or on CPU for the small example. If you are on Colab, please switch the runtime to GPU.


In [ ]:
# --- Install dependencies (uncomment on first run / Colab) ---
# !pip install -q torch torchvision
# !pip install -q open_clip_torch timm einops
# !pip install -q scanpy anndata scikit-learn umap-learn
# !pip install -q matplotlib seaborn pillow tqdm
# !pip install -q git+https://github.com/GuangyuWangLab2021/Loki.git  # OmiCLIP weights are distributed with Loki


In [ ]:
import os, sys, math, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn.functional as F

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'Torch version: {torch.__version__}')


## 2. Download a small demo dataset

We use a publicly available **10x Visium human breast cancer** slide subsampled to ~1,000 spots so the notebook runs end-to-end on CPU in a few minutes. The full pretraining corpus is much larger and is described in the slides.

If you cannot download data live, the `data/demo_visium/` folder shipped with this repository already contains:

* `tissue_image.tif` — the H&E whole-slide image (downsampled).
* `spots.csv` — pixel coordinates and tissue type for each spot.
* `counts.h5ad` — filtered gene-expression `AnnData`.


In [ ]:
DATA_DIR = Path('../data/demo_visium')
assert DATA_DIR.exists(), f'Demo data not found at {DATA_DIR}. See README for download instructions.'

import scanpy as sc
adata = sc.read_h5ad(DATA_DIR / 'counts.h5ad')
spots = pd.read_csv(DATA_DIR / 'spots.csv')

print(adata)
spots.head()


## 3. Load the pretrained OmiCLIP model

OmiCLIP weights ship inside the **Loki** package. There are three checkpoint sizes; here we use the base model.


In [ ]:
from loki.models import load_omiclip  # convenience wrapper

model, preprocess_image, tokenize_expression = load_omiclip(
    name='omiclip-base',     # also: 'omiclip-small', 'omiclip-large'
    device=DEVICE,
    pretrained=True,
)
model.eval();
print('Image encoder  :', model.visual.__class__.__name__)
print('Text/expr enc. :', model.transformer.__class__.__name__)
print('Embed dim      :', model.embed_dim)


### A quick look at the two encoders

* **Image encoder** — Vision Transformer (ViT-B/16) initialized from a histology-foundation backbone (CONCH/UNI-style). Outputs a 512-d (base) or 768-d (large) global token.
* **Transcriptomics encoder** — a 6-layer Transformer that consumes a **rank-ordered gene token sequence**: for each spot we take the top-256 expressed genes, sort them by expression, and feed them as tokens with learned embeddings. This is the same idea as scGPT / Geneformer but adapted for spot-level Visium data.

Both encoders end with a linear projection head into the shared 512-d space; embeddings are L2-normalized.


## 4. Prepare paired patches and expression tokens


In [ ]:
PATCH_SIZE = 224  # pixels
wsi = Image.open(DATA_DIR / 'tissue_image.tif').convert('RGB')
wsi_np = np.array(wsi)
print('WSI shape:', wsi_np.shape)

def crop_patch(img_np, x_px, y_px, size=PATCH_SIZE):
    half = size // 2
    x0, y0 = int(x_px) - half, int(y_px) - half
    x1, y1 = x0 + size, y0 + size
    # clip & pad if at the boundary
    patch = img_np[max(0, y0):y1, max(0, x0):x1]
    if patch.shape[:2] != (size, size):
        pad = np.zeros((size, size, 3), dtype=np.uint8)
        pad[:patch.shape[0], :patch.shape[1]] = patch
        patch = pad
    return patch

# show a few example patches
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (_, row) in zip(axes, spots.sample(5, random_state=0).iterrows()):
    patch = crop_patch(wsi_np, row['pxl_col'], row['pxl_row'])
    ax.imshow(patch); ax.set_title(row['region']); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def encode_spots(adata, spots, model, batch_size=64):
    """Return (img_emb, expr_emb) for every spot."""
    img_embs, expr_embs = [], []
    n = len(spots)
    for i in tqdm(range(0, n, batch_size), desc='Encoding'):
        idx = spots.index[i:i+batch_size]

        # --- image side
        patches = [
            preprocess_image(Image.fromarray(crop_patch(wsi_np, r['pxl_col'], r['pxl_row'])))
            for _, r in spots.loc[idx].iterrows()
        ]
        img_batch = torch.stack(patches).to(DEVICE)
        v = model.encode_image(img_batch)
        img_embs.append(F.normalize(v, dim=-1).cpu())

        # --- expression side
        expr_tokens = tokenize_expression(adata[idx])
        expr_tokens = {k: v.to(DEVICE) for k, v in expr_tokens.items()}
        t = model.encode_expression(**expr_tokens)
        expr_embs.append(F.normalize(t, dim=-1).cpu())

    return torch.cat(img_embs), torch.cat(expr_embs)

img_emb, expr_emb = encode_spots(adata, spots, model)
print('Image embeddings :', img_emb.shape)
print('Expr embeddings  :', expr_emb.shape)


## 5. Check the contrastive alignment

If pretraining worked, the diagonal of the cosine-similarity matrix between paired image and expression embeddings should be noticeably higher than the off-diagonal entries.


In [ ]:
sub = 300  # subsample for a readable matrix
idx = np.random.choice(len(img_emb), sub, replace=False)
S = (img_emb[idx] @ expr_emb[idx].T).numpy()

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(S, vmin=-0.4, vmax=0.4, cmap='RdBu_r')
ax.set_xlabel('Expression spot'); ax.set_ylabel('Image patch')
ax.set_title('Cosine similarity (paired ↔ unpaired)')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

diag = np.diag(S).mean()
offd = (S.sum() - np.trace(S)) / (sub*(sub-1))
print(f'mean diagonal  = {diag: .4f}')
print(f'mean off-diag. = {offd: .4f}')


## 6. Cross-modal retrieval

Given a query patch from the slide, OmiCLIP can rank all spots by the cosine similarity between their expression embeddings and the query image embedding — and vice versa. This is the simplest downstream use of the model and the building block of every Loki/Thor workflow.


In [ ]:
def retrieve(query_emb, target_emb, k=5):
    sims = (query_emb @ target_emb.T).squeeze()
    top_idx = torch.topk(sims, k).indices.cpu().numpy()
    return top_idx, sims[top_idx].cpu().numpy()

q = 17
top_idx, top_sim = retrieve(img_emb[q:q+1], expr_emb, k=5)
print(f'Query patch index : {q} ({spots.iloc[q]["region"]})')
print('Top-5 retrieved spots (by transcriptomics):')
for i, s in zip(top_idx, top_sim):
    print(f'  spot {i:>4d}  sim={s:.3f}  region={spots.iloc[i]["region"]}')


In [ ]:
# Visual sanity check: the query and its top transcriptomic matches should look similar
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(crop_patch(wsi_np, spots.iloc[q]['pxl_col'], spots.iloc[q]['pxl_row']))
axes[0].set_title('Query'); axes[0].axis('off')
for ax, i, s in zip(axes[1:], top_idx, top_sim):
    ax.imshow(crop_patch(wsi_np, spots.iloc[i]['pxl_col'], spots.iloc[i]['pxl_row']))
    ax.set_title(f'sim={s:.2f}'); ax.axis('off')
plt.tight_layout(); plt.show()


## 7. Visualizing the joint embedding space


In [ ]:
import umap

all_emb = torch.cat([img_emb, expr_emb]).numpy()
labels  = np.array(['image']*len(img_emb) + ['expr']*len(expr_emb))
regions = np.tile(spots['region'].values, 2)

reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=0)
uxy = reducer.fit_transform(all_emb)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, color_by, vals, title in zip(
        axes,
        ['modality', 'region'],
        [labels, regions],
        ['Modality (image vs. expression)', 'Tissue region']):
    sns.scatterplot(x=uxy[:,0], y=uxy[:,1], hue=vals, s=10, ax=ax, linewidth=0)
    ax.set_title(title); ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()


### What to look for in the UMAP

* In the **left** panel the two modalities should be *interleaved* rather than separated — that is the visible signature of contrastive alignment.
* In the **right** panel, points should cluster by **tissue region**, not by modality. Image and expression embeddings of the same region land next to each other.

If you instead saw two clean clouds (one per modality), the model would have failed to align — a useful negative-control sanity check when training your own version.


## 8. (Optional) Inspect the contrastive loss on this slide

We can compute the held-out InfoNCE loss for a mini-batch of spots; lower is better.


In [ ]:
@torch.no_grad()
def info_nce(v, t, temperature=0.07):
    logits = (v @ t.T) / temperature
    targets = torch.arange(len(v), device=v.device)
    return 0.5*(F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets))

bs = 128
idx = np.random.choice(len(img_emb), bs, replace=False)
loss = info_nce(img_emb[idx], expr_emb[idx])
print(f'Symmetric InfoNCE on {bs} spots: {loss.item():.4f}')
print('(Random baseline ≈ ln(bs) = %.3f)' % math.log(bs))


## 9. Recap and what's next

You have just used OmiCLIP to:

* embed Visium spots in a shared image–expression space,
* perform cross-modal retrieval, and
* qualitatively verify that paired embeddings are aligned.

In **Part 4** we move from *embeddings* to *applications*. The Loki platform reuses these frozen encoders to tackle:

1. **Tissue alignment** across consecutive sections,
2. **Annotation** with bulk RNA-seq references or marker-gene panels,
3. **Cell-type decomposition** of each spot,
4. **Image-to-transcriptomics retrieval** at slide scale, and
5. **Gene-expression prediction directly from H&E** — bringing molecular maps to slides where no Visium data was ever collected.

The companion platform **Thor** scales these tasks to whole-slide and multi-slide analyses.

👉 Continue with `Part4_Loki_Thor_Applications.ipynb`.
